# 10 — v11: Price Interaction Target Encoding

**Current best:** v10 (postal-sector smoothing) → Kaggle RMSE **~21,755** | Goal: push below $21,500.

## What changes vs v10

| # | Change | Why |
|---|---|---|
| 1 | **Fix `hdb_age`**: recompute as `Tranc_Year - lease_commence_date` | Raw data uses fixed 2021 reference — inconsistent with `remaining_lease` (which uses `Tranc_Year`). Now `hdb_age + remaining_lease = 99` always. |
| 2 | **`ft_sector_median_price`**: OOF median by `(flat_type, postal_sector)` | Same sector, different flat type = up to $595K gap. Current `postal_sector_median_price` flattens this. Standalone R²=0.86. |
| 3 | **`yr_sector_median_price`**: OOF median by `(Tranc_Year, postal_sector)` | Prices within a sector are not static 2012–2021. Provides time-adjusted geographic price signal. |
| 4 | **`yr_flattype_median_price`**: OOF median by `(Tranc_Year, flat_type)` | Different flat types appreciated at different rates. 70 fully-populated groups, zero sparsity risk. |

All other settings (model params, stacking architecture) are identical to v10.

---
## 1. Imports & Load Data

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from math import radians, cos, sin, asin, sqrt

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split, KFold, RandomizedSearchCV
from sklearn.metrics import mean_squared_error, make_scorer

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

train = pd.read_csv('../../data/raw/train.csv', low_memory=False)
test  = pd.read_csv('../../data/raw/test.csv',  low_memory=False)
print(f'Train: {train.shape}  |  Test: {test.shape}')

Train: (150634, 77)  |  Test: (16737, 76)


---
## 2. Feature Engineering

Changes vs v10:
- **`hdb_age` fix**: now `Tranc_Year - lease_commence_date` (was hardcoded to 2021 reference in raw data). Ensures `hdb_age + remaining_lease = 99`.
- **3 composite key columns** for new OOF encodings: `ft_sector_key`, `yr_sector_key`, `yr_ft_key`.

In [2]:
DROP_COLS   = ['floor_area_sqft','lower','upper','mid','full_flat_type',
               'address','Tranc_YearMonth','residential','year_completed']
SOLD_COLS   = ['1room_sold','2room_sold','3room_sold','4room_sold',
               '5room_sold','exec_sold','multigen_sold','studio_apartment_sold']
RENTAL_COLS = ['1room_rental','2room_rental','3room_rental','other_room_rental']
FILL_ZERO   = ['Hawker_Within_500m','Mall_Within_500m','Hawker_Within_1km',
               'Mall_Within_1km','Hawker_Within_2km','Mall_Within_2km']
MATURE_ESTATES = {
    'ANG MO KIO','BEDOK','BISHAN','BUKIT MERAH','BUKIT TIMAH',
    'CENTRAL AREA','CLEMENTI','GEYLANG','KALLANG/WHAMPOA',
    'MARINE PARADE','PASIR RIS','QUEENSTOWN','SERANGOON','TAMPINES','TOA PAYOH'
}
ROOM_COUNT = {'1 ROOM':1,'2 ROOM':2,'3 ROOM':3,'4 ROOM':4,
              '5 ROOM':5,'EXECUTIVE':5,'MULTI-GENERATION':6}
CBD_LAT, CBD_LON = 1.2847, 103.8510

STREET_FREQ = train['street_name'].value_counts().to_dict()

def haversine_km(lat, lon, lat2=CBD_LAT, lon2=CBD_LON):
    R = 6371
    lat, lon, lat2, lon2 = map(radians, [lat, lon, lat2, lon2])
    a = sin((lat2-lat)/2)**2 + cos(lat)*cos(lat2)*sin((lon2-lon)/2)**2
    return 2*R*asin(sqrt(a))

def engineer_features(df, amenity_caps=None):
    df = df.copy()
    for c in FILL_ZERO:
        df[c] = df[c].fillna(0)

    # --- Lease features ---
    # FIX: recompute hdb_age using Tranc_Year so hdb_age + remaining_lease = 99
    df['remaining_lease'] = 99 - (df['Tranc_Year'] - df['lease_commence_date'])
    df['hdb_age']         = df['Tranc_Year'] - df['lease_commence_date']

    # --- Existing engineered features (unchanged from v10) ---
    df['dist_to_cbd']      = df.apply(lambda r: haversine_km(r['Latitude'], r['Longitude']), axis=1)
    df['is_mature_estate'] = df['town'].str.upper().isin(MATURE_ESTATES).astype(int)
    df['tranc_month_sin']     = np.sin(2*np.pi*df['Tranc_Month']/12)
    df['tranc_month_cos']     = np.cos(2*np.pi*df['Tranc_Month']/12)
    df['total_sold']          = df[SOLD_COLS].sum(axis=1)
    df['total_rental']        = df[RENTAL_COLS].sum(axis=1)
    df['rental_ratio']        = (df['total_rental'] / df['total_dwelling_units'].replace(0, np.nan)).fillna(0)
    df['num_rooms']           = df['flat_type'].str.upper().map(ROOM_COUNT).fillna(4)
    df['floor_area_per_room'] = df['floor_area_sqm'] / df['num_rooms']
    caps = amenity_caps or {}
    new_caps = {}
    for col in ['mrt_nearest_distance','Mall_Nearest_Distance','Hawker_Nearest_Distance']:
        cap = caps.get(col, df[col].quantile(0.99))
        new_caps[col] = cap
        inv = 1 / df[col].clip(1, cap)
        inv_min, inv_max = inv.min(), inv.max()
        df[f'_inv_{col}'] = (inv - inv_min) / (inv_max - inv_min + 1e-9)
    df['amenity_score'] = (df['_inv_mrt_nearest_distance'] +
                           df['_inv_Mall_Nearest_Distance'] +
                           df['_inv_Hawker_Nearest_Distance']) / 3
    df.drop(columns=[c for c in df.columns if c.startswith('_inv_')], inplace=True)
    df['dist_x_storey']   = df['dist_to_cbd'] * df['mid_storey']
    df['lease_x_area']    = df['remaining_lease'] * df['floor_area_sqm']
    df['log_dist_to_cbd'] = np.log1p(df['dist_to_cbd'])
    df['street_freq']     = df['street_name'].map(STREET_FREQ).fillna(0)
    df['postal_sector']   = df['postal'].astype(str).str[:4]
    df['block_num']       = pd.to_numeric(
        df['block'].astype(str).str.extract(r'(\d+)')[0], errors='coerce'
    ).fillna(0)

    # --- NEW: composite keys for interaction OOF encoding ---
    df['ft_sector_key'] = df['flat_type'].astype(str) + '_' + df['postal_sector'].astype(str)
    df['yr_sector_key'] = df['Tranc_Year'].astype(str) + '_' + df['postal_sector'].astype(str)
    df['yr_ft_key']     = df['Tranc_Year'].astype(str) + '_' + df['flat_type'].astype(str)

    return df, new_caps

train_fe, train_caps = engineer_features(train)
test_fe,  _          = engineer_features(test, amenity_caps=train_caps)
print(f'Train: {train_fe.shape}  |  Test: {test_fe.shape}')

# Verify hdb_age fix: hdb_age + remaining_lease should always equal 99
check = (train_fe['hdb_age'] + train_fe['remaining_lease'] == 99).all()
print(f'hdb_age + remaining_lease == 99 for all rows: {check}')  # must be True
print(f'ft_sector_key unique (train): {train_fe["ft_sector_key"].nunique()}')
print(f'yr_sector_key unique (train): {train_fe["yr_sector_key"].nunique()}')
print(f'yr_ft_key unique (train):     {train_fe["yr_ft_key"].nunique()}')

Train: (150634, 97)  |  Test: (16737, 96)
hdb_age + remaining_lease == 99 for all rows: True
ft_sector_key unique (train): 1772
yr_sector_key unique (train): 5133
yr_ft_key unique (train):     70


---
## 3. Per-Fold OOF Target Encoding

**3 existing encodings** (unchanged from v10) + **3 new interaction encodings**:

| Group key | Output column | min_count | Fallback |
|---|---|---|---|
| `town` | `town_median_price` | 1 | global median |
| `postal_sector` | `postal_sector_median_price` | 10 | global median |
| `flat_model` | `flat_model_median_price` | 1 | global median |
| `ft_sector_key` | `ft_sector_median_price` | 10 | `postal_sector_median_price` |
| `yr_sector_key` | `yr_sector_median_price` | 10 | `postal_sector_median_price` |
| `yr_ft_key` | `yr_flattype_median_price` | 1 | global median |

> For the two high-sparsity interaction features (`ft_sector_key`, `yr_sector_key`), sparse groups fall back to the parent `postal_sector_median_price` (a tighter fallback than global median).

In [3]:
MIN_COUNT_SECTOR = 10
MIN_COUNT_INTER  = 10

def oof_group_median(group_series, y_series, n_splits=5, random_state=42, min_count=1):
    """OOF median target encoding. Groups with < min_count rows fall back to global median."""
    groups = group_series.values
    y      = y_series.values
    encoded    = np.zeros(len(groups))
    global_med = np.median(y)
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    for tr_idx, va_idx in kf.split(groups):
        fold_counts = {}
        fold_vals   = {}
        for g, p in zip(groups[tr_idx], y[tr_idx]):
            fold_counts[g] = fold_counts.get(g, 0) + 1
            fold_vals.setdefault(g, []).append(p)
        fold_map = {g: np.median(ps) for g, ps in fold_vals.items()
                    if fold_counts[g] >= min_count}
        for i in va_idx:
            encoded[i] = fold_map.get(groups[i], global_med)
    return encoded

# --- Training set: OOF encoding (no leakage) ---
train_fe['town_median_price']          = oof_group_median(train_fe['town'],          train['resale_price'])
train_fe['postal_sector_median_price'] = oof_group_median(train_fe['postal_sector'], train['resale_price'],
                                                           min_count=MIN_COUNT_SECTOR)
train_fe['flat_model_median_price']    = oof_group_median(train_fe['flat_model'],    train['resale_price'])
train_fe['ft_sector_median_price']     = oof_group_median(train_fe['ft_sector_key'], train['resale_price'],
                                                           min_count=MIN_COUNT_INTER)
train_fe['yr_sector_median_price']     = oof_group_median(train_fe['yr_sector_key'], train['resale_price'],
                                                           min_count=MIN_COUNT_INTER)
train_fe['yr_flattype_median_price']   = oof_group_median(train_fe['yr_ft_key'],     train['resale_price'])

# --- Test set: full-training-set maps ---
_tmp = pd.DataFrame({
    'town':          train_fe['town'].values,
    'postal_sector': train_fe['postal_sector'].values,
    'flat_model':    train_fe['flat_model'].values,
    'ft_sector_key': train_fe['ft_sector_key'].values,
    'yr_sector_key': train_fe['yr_sector_key'].values,
    'yr_ft_key':     train_fe['yr_ft_key'].values,
    'resale_price':  train['resale_price'].values,
})

full_town_map   = _tmp.groupby('town')['resale_price'].median()

sector_counts   = _tmp.groupby('postal_sector')['resale_price'].count()
valid_sectors   = sector_counts[sector_counts >= MIN_COUNT_SECTOR].index
full_sector_map = _tmp[_tmp['postal_sector'].isin(valid_sectors)].groupby('postal_sector')['resale_price'].median()

full_model_map  = _tmp.groupby('flat_model')['resale_price'].median()

# ft_sector: groups with <10 rows fall back to parent postal_sector map
ft_sector_counts  = _tmp.groupby('ft_sector_key')['resale_price'].count()
valid_ft_sectors  = ft_sector_counts[ft_sector_counts >= MIN_COUNT_INTER].index
full_ft_sector_map = _tmp[_tmp['ft_sector_key'].isin(valid_ft_sectors)].groupby('ft_sector_key')['resale_price'].median()

# yr_sector: groups with <10 rows fall back to parent postal_sector map
yr_sector_counts  = _tmp.groupby('yr_sector_key')['resale_price'].count()
valid_yr_sectors  = yr_sector_counts[yr_sector_counts >= MIN_COUNT_INTER].index
full_yr_sector_map = _tmp[_tmp['yr_sector_key'].isin(valid_yr_sectors)].groupby('yr_sector_key')['resale_price'].median()

# yr_ft: all 70 groups populated, use global median fallback
full_yr_ft_map  = _tmp.groupby('yr_ft_key')['resale_price'].median()
global_med_train = train['resale_price'].median()

# Apply to test
test_fe['town_median_price']          = test_fe['town'].map(full_town_map).fillna(full_town_map.median())
test_fe['postal_sector_median_price'] = test_fe['postal_sector'].map(full_sector_map).fillna(full_sector_map.median())
test_fe['flat_model_median_price']    = test_fe['flat_model'].map(full_model_map).fillna(full_model_map.median())

# ft_sector fallback: unmapped keys → use the parent postal_sector map
test_fe['ft_sector_median_price'] = test_fe['ft_sector_key'].map(full_ft_sector_map)
mask = test_fe['ft_sector_median_price'].isna()
test_fe.loc[mask, 'ft_sector_median_price'] = test_fe.loc[mask, 'postal_sector'].map(full_sector_map).fillna(full_sector_map.median())

# yr_sector fallback: unmapped keys → use the parent postal_sector map
test_fe['yr_sector_median_price'] = test_fe['yr_sector_key'].map(full_yr_sector_map)
mask = test_fe['yr_sector_median_price'].isna()
test_fe.loc[mask, 'yr_sector_median_price'] = test_fe.loc[mask, 'postal_sector'].map(full_sector_map).fillna(full_sector_map.median())

# yr_ft fallback: global median (all groups populated)
test_fe['yr_flattype_median_price'] = test_fe['yr_ft_key'].map(full_yr_ft_map).fillna(global_med_train)

print('OOF encoding done.')
for col in ['town_median_price','postal_sector_median_price','flat_model_median_price',
            'ft_sector_median_price','yr_sector_median_price','yr_flattype_median_price']:
    print(f'  {col}: train_null={train_fe[col].isna().sum()}, test_null={test_fe[col].isna().sum()}')

OOF encoding done.
  town_median_price: train_null=0, test_null=0
  postal_sector_median_price: train_null=0, test_null=0
  flat_model_median_price: train_null=0, test_null=0
  ft_sector_median_price: train_null=0, test_null=0
  yr_sector_median_price: train_null=0, test_null=0
  yr_flattype_median_price: train_null=0, test_null=0


---
## 4. Prepare X and y

Drop: raw `postal`, `block` (replaced by encoded versions), composite key columns (used for encoding only, not as model features), and all other redundant columns.

In [4]:
# Composite key columns served their purpose in encoding — drop them as model inputs
KEY_COLS = ['ft_sector_key', 'yr_sector_key', 'yr_ft_key', 'postal_sector']

DROP_ALL = (['id', 'resale_price', 'postal', 'block']
            + DROP_COLS + SOLD_COLS + RENTAL_COLS + ['num_rooms']
            + KEY_COLS)

X      = train_fe.drop(columns=DROP_ALL, errors='ignore')
y      = train['resale_price'].values
y_log  = np.log1p(y)

X_test = test_fe.drop(columns=[c for c in DROP_ALL if c != 'resale_price'], errors='ignore')
X_test = X_test.reindex(columns=X.columns, fill_value=0)

num_cols = X.select_dtypes(include='number').columns.tolist()
cat_cols = X.select_dtypes(include='object').columns.tolist()

for col in cat_cols:
    X[col]      = X[col].astype(str)
    X_test[col] = X_test[col].astype(str)

print(f'Features: {X.shape[1]}  (num={len(num_cols)}, cat={len(cat_cols)})')
print(f'New features present: ft_sector={"ft_sector_median_price" in num_cols}, '
      f'yr_sector={"yr_sector_median_price" in num_cols}, '
      f'yr_ft={"yr_flattype_median_price" in num_cols}')
assert X_test.shape[1] == X.shape[1], 'X/X_test column mismatch'

Features: 73  (num=59, cat=14)
New features present: ft_sector=True, yr_sector=True, yr_ft=True


---
## 5. Preprocessing Pipeline

In [5]:
num_pipe = Pipeline([('imp', SimpleImputer(strategy='median'))])
cat_pipe = Pipeline([
    ('imp', SimpleImputer(strategy='most_frequent')),
    ('enc', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])
preprocessor = ColumnTransformer([
    ('num', num_pipe, num_cols),
    ('cat', cat_pipe, cat_cols)
])

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def neg_dollar_rmse(y_log_true, y_log_pred):
    return -np.sqrt(mean_squared_error(np.expm1(y_log_true), np.expm1(y_log_pred)))

dollar_rmse_scorer = make_scorer(neg_dollar_rmse)
print('Preprocessor ready.')

Preprocessor ready.


---
## 6. RandomizedSearchCV — XGBoost

In [6]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
y_train_log = np.log1p(y_train)
y_val_log   = np.log1p(y_val)

param_dist_xgb = {
    'model__n_estimators'     : [500, 800, 1000, 1500, 2000],
    'model__max_depth'        : [4, 5, 6, 7, 8, 9],
    'model__learning_rate'    : [0.01, 0.03, 0.05, 0.07, 0.1],
    'model__subsample'        : [0.6, 0.7, 0.8, 0.9],
    'model__colsample_bytree' : [0.4, 0.5, 0.6, 0.7, 0.8],
    'model__min_child_weight' : [1, 3, 5, 7, 10],
    'model__reg_alpha'        : [0, 0.01, 0.1, 1.0],
    'model__reg_lambda'       : [0.5, 1.0, 1.5, 2.0, 3.0],
}
xgb_search = RandomizedSearchCV(
    Pipeline([('pre', preprocessor),
              ('model', XGBRegressor(random_state=42, n_jobs=-1, verbosity=0, tree_method='hist'))]),
    param_distributions=param_dist_xgb, n_iter=40, cv=3,
    scoring=dollar_rmse_scorer, random_state=42, n_jobs=-1, verbose=1, refit=True,
)
xgb_search.fit(X_train, y_train_log)
print(f'\nXGB best CV RMSE: ${-xgb_search.best_score_:,.0f}')
print('Best params:', xgb_search.best_params_)

XGB_PARAMS = {k.replace('model__',''):v for k,v in xgb_search.best_params_.items()}
XGB_PARAMS.update({'random_state':42, 'n_jobs':-1, 'verbosity':0, 'tree_method':'hist'})

xgb_check = Pipeline([('pre', preprocessor), ('model', XGBRegressor(**XGB_PARAMS))])
xgb_check.fit(X_train, y_train_log)
xgb_val_rmse = rmse(y_val, np.expm1(xgb_check.predict(X_val)))
print(f'XGB val RMSE: ${xgb_val_rmse:,.0f}')

Fitting 3 folds for each of 40 candidates, totalling 120 fits

XGB best CV RMSE: $22,766
Best params: {'model__subsample': 0.9, 'model__reg_lambda': 1.5, 'model__reg_alpha': 0.01, 'model__n_estimators': 2000, 'model__min_child_weight': 7, 'model__max_depth': 7, 'model__learning_rate': 0.05, 'model__colsample_bytree': 0.4}
XGB val RMSE: $21,887


---
## 7. RandomizedSearchCV — LightGBM

In [7]:
param_dist_lgbm = {
    'model__n_estimators'     : [500, 800, 1000, 1500, 2000],
    'model__max_depth'        : [5, 6, 7, 8, 10, 12],
    'model__num_leaves'       : [60, 80, 100, 127, 200, 300],
    'model__learning_rate'    : [0.01, 0.03, 0.05, 0.07, 0.1],
    'model__subsample'        : [0.6, 0.7, 0.8, 0.9],
    'model__colsample_bytree' : [0.5, 0.6, 0.7, 0.8],
    'model__min_child_samples': [10, 20, 30, 50],
    'model__reg_alpha'        : [0, 0.01, 0.1, 1.0],
    'model__reg_lambda'       : [0, 0.1, 0.5, 1.0],
}
lgbm_search = RandomizedSearchCV(
    Pipeline([('pre', preprocessor),
              ('model', LGBMRegressor(random_state=42, n_jobs=-1, verbosity=-1))]),
    param_distributions=param_dist_lgbm, n_iter=40, cv=3,
    scoring=dollar_rmse_scorer, random_state=42, n_jobs=-1, verbose=1, refit=True,
)
lgbm_search.fit(X_train, y_train_log)
print(f'\nLGBM best CV RMSE: ${-lgbm_search.best_score_:,.0f}')
print('Best params:', lgbm_search.best_params_)

LGBM_PARAMS = {k.replace('model__',''):v for k,v in lgbm_search.best_params_.items()}
LGBM_PARAMS.update({'random_state':42, 'n_jobs':-1, 'verbosity':-1})

lgbm_check = Pipeline([('pre', preprocessor), ('model', LGBMRegressor(**LGBM_PARAMS))])
lgbm_check.fit(X_train, y_train_log)
lgbm_val_rmse = rmse(y_val, np.expm1(lgbm_check.predict(X_val)))
print(f'LGBM val RMSE: ${lgbm_val_rmse:,.0f}')

Fitting 3 folds for each of 40 candidates, totalling 120 fits

LGBM best CV RMSE: $22,774
Best params: {'model__subsample': 0.8, 'model__reg_lambda': 0.5, 'model__reg_alpha': 0, 'model__num_leaves': 300, 'model__n_estimators': 1000, 'model__min_child_samples': 20, 'model__max_depth': 12, 'model__learning_rate': 0.03, 'model__colsample_bytree': 0.5}
LGBM val RMSE: $21,782


---
## 8. RandomizedSearchCV — Extra Trees

In [8]:
param_dist_et = {
    'model__n_estimators'     : [200, 300, 500],
    'model__max_depth'        : [15, 20, 30],
    'model__min_samples_split': [2, 5, 10],
    'model__min_samples_leaf' : [1, 2, 4],
    'model__max_features'     : [0.5, 0.6, 0.7, 0.8],
}
et_search = RandomizedSearchCV(
    Pipeline([('pre', preprocessor), ('model', ExtraTreesRegressor(random_state=42, n_jobs=-1))]),
    param_distributions=param_dist_et, n_iter=15, cv=3,
    scoring=dollar_rmse_scorer, random_state=42, n_jobs=-1, verbose=1, refit=True,
)
et_search.fit(X_train, y_train_log)
print(f'ET best CV RMSE: ${-et_search.best_score_:,.0f}')
et_val_rmse = rmse(y_val, np.expm1(et_search.best_estimator_.predict(X_val)))
print(f'ET val RMSE:     ${et_val_rmse:,.0f}')

ET_PARAMS = {k.replace('model__',''):v for k,v in et_search.best_params_.items()}
ET_PARAMS.update({'random_state':42,'n_jobs':-1})
print(f'ET_PARAMS: {ET_PARAMS}')

Fitting 3 folds for each of 15 candidates, totalling 45 fits
ET best CV RMSE: $24,490
ET val RMSE:     $23,339
ET_PARAMS: {'n_estimators': 200, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_features': 0.5, 'max_depth': 30, 'random_state': 42, 'n_jobs': -1}


---
## 9. Generate OOF Predictions (5-Fold) — 3 Models

All 6 target encodings are **recomputed inside each fold** from fold-train rows only — no leakage.

For the two sparse interaction keys (`ft_sector_key`, `yr_sector_key`), groups with < `MIN_COUNT_INTER` rows fall back to the parent `postal_sector` map for that fold.

In [11]:
kf5 = KFold(n_splits=5, shuffle=True, random_state=42)

xgb_oof  = np.zeros(len(X))
lgbm_oof = np.zeros(len(X))
et_oof   = np.zeros(len(X))

xgb_test_folds  = np.zeros((5, len(X_test)))
lgbm_test_folds = np.zeros((5, len(X_test)))
et_test_folds   = np.zeros((5, len(X_test)))

fold_rmses_xgb  = []
fold_rmses_lgbm = []
fold_rmses_et   = []

# (group_col, price_col, min_count, fallback_col_or_None)
# fallback_col: for sparse interaction keys, fall back to the parent postal_sector encoding
ENCODE_PAIRS = [
    ('town',          'town_median_price',          1,                None),
    ('postal_sector', 'postal_sector_median_price', MIN_COUNT_SECTOR, None),
    ('flat_model',    'flat_model_median_price',    1,                None),
    ('ft_sector_key', 'ft_sector_median_price',     MIN_COUNT_INTER,  'postal_sector_median_price'),
    ('yr_sector_key', 'yr_sector_median_price',     MIN_COUNT_INTER,  'postal_sector_median_price'),
    ('yr_ft_key',     'yr_flattype_median_price',   1,                None),
]

print('Generating OOF predictions (5-fold, 3 models)...')
print(f'{"Fold":>5}  {"XGB RMSE":>12}  {"LGBM RMSE":>12}  {"ET RMSE":>12}')
print('-' * 55)

for fold, (tr_idx, va_idx) in enumerate(kf5.split(X)):
    X_tr = X.iloc[tr_idx].copy()
    X_va = X.iloc[va_idx].copy()
    y_tr, y_va = y[tr_idx], y[va_idx]
    y_tr_log      = np.log1p(y_tr)
    global_med_tr = np.median(y_tr)

    # Recompute all target encodings per-fold (prevents leakage).
    # Group key cols were dropped from X — read them from train_fe instead.
    for group_col, price_col, min_ct, fallback_col in ENCODE_PAIRS:
        tr_groups = train_fe.iloc[tr_idx][group_col].values
        va_groups = train_fe.iloc[va_idx][group_col].values

        fold_counts = {}
        fold_vals   = {}
        for g, p in zip(tr_groups, y_tr):
            fold_counts[g] = fold_counts.get(g, 0) + 1
            fold_vals.setdefault(g, []).append(p)
        fold_map = {g: np.median(ps) for g, ps in fold_vals.items()
                    if fold_counts[g] >= min_ct}

        if fallback_col is None:
            X_tr[price_col] = pd.Series(tr_groups, index=X_tr.index).map(fold_map).fillna(global_med_tr)
            X_va[price_col] = pd.Series(va_groups, index=X_va.index).map(fold_map).fillna(global_med_tr)
        else:
            # Sparse interaction: fall back to parent encoding (already set earlier in this loop)
            tr_mapped = pd.Series(tr_groups, index=X_tr.index).map(fold_map)
            mask_tr = tr_mapped.isna()
            tr_mapped.loc[mask_tr] = X_tr.loc[mask_tr, fallback_col]
            X_tr[price_col] = tr_mapped.fillna(global_med_tr)

            va_mapped = pd.Series(va_groups, index=X_va.index).map(fold_map)
            mask_va = va_mapped.isna()
            va_mapped.loc[mask_va] = X_va.loc[mask_va, fallback_col]
            X_va[price_col] = va_mapped.fillna(global_med_tr)

    # XGBoost
    xgb_pipe = Pipeline([('pre', preprocessor), ('model', XGBRegressor(**XGB_PARAMS))])
    xgb_pipe.fit(X_tr, y_tr_log)
    xgb_oof[va_idx]      = xgb_pipe.predict(X_va)
    xgb_test_folds[fold] = xgb_pipe.predict(X_test)
    fold_rmses_xgb.append(rmse(y_va, np.expm1(xgb_oof[va_idx])))

    # LightGBM
    lgbm_pipe = Pipeline([('pre', preprocessor), ('model', LGBMRegressor(**LGBM_PARAMS))])
    lgbm_pipe.fit(X_tr, y_tr_log)
    lgbm_oof[va_idx]      = lgbm_pipe.predict(X_va)
    lgbm_test_folds[fold] = lgbm_pipe.predict(X_test)
    fold_rmses_lgbm.append(rmse(y_va, np.expm1(lgbm_oof[va_idx])))

    # Extra Trees
    et_pipe = Pipeline([('pre', preprocessor), ('model', ExtraTreesRegressor(**ET_PARAMS))])
    et_pipe.fit(X_tr, y_tr_log)
    et_oof[va_idx]      = et_pipe.predict(X_va)
    et_test_folds[fold] = et_pipe.predict(X_test)
    fold_rmses_et.append(rmse(y_va, np.expm1(et_oof[va_idx])))

    print(f'{fold+1:>5}  ${fold_rmses_xgb[-1]:>10,.0f}  ${fold_rmses_lgbm[-1]:>10,.0f}  ${fold_rmses_et[-1]:>10,.0f}')

print('-' * 55)
print(f'{"Mean":>5}  ${np.mean(fold_rmses_xgb):>10,.0f}  ${np.mean(fold_rmses_lgbm):>10,.0f}  ${np.mean(fold_rmses_et):>10,.0f}')

xgb_test_avg  = xgb_test_folds.mean(axis=0)
lgbm_test_avg = lgbm_test_folds.mean(axis=0)
et_test_avg   = et_test_folds.mean(axis=0)

print('\nOOF generation complete.')

Generating OOF predictions (5-fold, 3 models)...
 Fold      XGB RMSE     LGBM RMSE       ET RMSE
-------------------------------------------------------
    1  $    21,593  $    21,642  $    23,399
    2  $    22,460  $    22,515  $    23,980
    3  $    21,738  $    21,727  $    23,594
    4  $    22,031  $    22,024  $    23,701
    5  $    21,933  $    21,803  $    23,550
-------------------------------------------------------
 Mean  $    21,951  $    21,942  $    23,645

OOF generation complete.


---
## 10. Sanity Check — Individual Models & Blends

In [12]:
print('Individual OOF RMSE:')
print(f'  XGB:  ${rmse(y, np.expm1(xgb_oof)):>8,.0f}')
print(f'  LGBM: ${rmse(y, np.expm1(lgbm_oof)):>8,.0f}')
print(f'  ET:   ${rmse(y, np.expm1(et_oof)):>8,.0f}')

blend_equal = np.expm1((xgb_oof + lgbm_oof + et_oof) / 3)
print(f'\n3-model equal-weight blend OOF RMSE: ${rmse(y, blend_equal):>8,.0f}')
print(f'v10 Kaggle RMSE (for comparison):      ~$21,755')

Individual OOF RMSE:
  XGB:  $  21,953
  LGBM: $  21,944
  ET:   $  23,645

3-model equal-weight blend OOF RMSE: $  21,846
v10 Kaggle RMSE (for comparison):      ~$21,755


---
## 11. Ridge Meta-Model on 3 OOF Features

In [13]:
meta_X_train = np.column_stack([xgb_oof, lgbm_oof, et_oof])
meta_X_test  = np.column_stack([xgb_test_avg, lgbm_test_avg, et_test_avg])

print('Ridge alpha sweep (3 meta-features):')
print(f'{"alpha":>10}  {"RMSE":>12}  {"XGB coef":>10}  {"LGBM coef":>10}  {"ET coef":>10}')
print('-' * 62)

best_meta_rmse   = float('inf')
best_alpha_ridge = 1.0

for alpha in [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]:
    ridge = Ridge(alpha=alpha)
    ridge.fit(meta_X_train, y_log)
    meta_pred_log = ridge.predict(meta_X_train)
    meta_rmse_val = rmse(y, np.expm1(meta_pred_log))
    print(f'{alpha:>10.3f}  ${meta_rmse_val:>10,.0f}  {ridge.coef_[0]:>10.4f}  {ridge.coef_[1]:>10.4f}  {ridge.coef_[2]:>10.4f}')
    if meta_rmse_val < best_meta_rmse:
        best_meta_rmse   = meta_rmse_val
        best_alpha_ridge = alpha

print(f'\nBest Ridge alpha: {best_alpha_ridge}')
meta_model = Ridge(alpha=best_alpha_ridge)
meta_model.fit(meta_X_train, y_log)
print(f'Learned coefficients:')
print(f'  XGB:  {meta_model.coef_[0]:.4f}')
print(f'  LGBM: {meta_model.coef_[1]:.4f}')
print(f'  ET:   {meta_model.coef_[2]:.4f}')
print(f'  Intercept: {meta_model.intercept_:.4f}')
print(f'\nOOF meta-train RMSE (optimistic): ${best_meta_rmse:,.0f}')
print(f'v10 Kaggle RMSE for comparison:     ~$21,755')

Ridge alpha sweep (3 meta-features):
     alpha          RMSE    XGB coef   LGBM coef     ET coef
--------------------------------------------------------------
     0.001  $    21,716      0.4524      0.4113      0.1389
     0.010  $    21,716      0.4524      0.4113      0.1389
     0.100  $    21,716      0.4521      0.4112      0.1393
     1.000  $    21,716      0.4498      0.4102      0.1426
    10.000  $    21,721      0.4318      0.4003      0.1704
   100.000  $    21,776      0.3731      0.3610      0.2666

Best Ridge alpha: 0.001
Learned coefficients:
  XGB:  0.4524
  LGBM: 0.4113
  ET:   0.1389
  Intercept: -0.0333

OOF meta-train RMSE (optimistic): $21,716
v10 Kaggle RMSE for comparison:     ~$21,755


---
## 12. Generate Submission v11

In [ ]:
final_log  = meta_model.predict(meta_X_test)
final_pred = np.expm1(final_log)

sample_sub = pd.read_csv('../../outputs/submissions/sample_sub_reg.csv')
sub_v11 = pd.DataFrame({'Id': test['id'], 'Predicted': final_pred})
sub_v11 = sub_v11.set_index('Id').reindex(sample_sub['Id']).reset_index()

out = '../../outputs/submissions/sub_v11_price_interactions.csv'
sub_v11.to_csv(out, index=False)
print(f'Saved: {out}')
print(f'Shape: {sub_v11.shape}')
print(f'Price range: ${sub_v11.Predicted.min():,.0f} – ${sub_v11.Predicted.max():,.0f}')
print(f'Price mean:  ${sub_v11.Predicted.mean():,.0f}')

---
## 13. Summary

### Changes vs v10
| Change | Why | Expected Gain |
|---|---|---|
| Fix `hdb_age` (use `Tranc_Year`) | Raw data used fixed 2021 reference — inconsistent with `remaining_lease`. Removes noise from contradictory features. | ~10–30 |
| `ft_sector_median_price` | Flat type × sector interaction. Standalone R²=0.86. Captures $595K gap between flat types in same sector. | ~100–200 |
| `yr_sector_median_price` | Time-adjusted sector price. Captures appreciation trends within sectors 2012–2021. | ~50–100 |
| `yr_flattype_median_price` | Flat-type-specific price trends by year. Zero sparsity (70 groups). | ~20–50 |

### Score tracker
| Version | Model | Kaggle RMSE |
|---|---|---|
| v8 | XGB+LGBM+ET stack | 21,804.67 |
| v9 | Wider search + postal_sector + flat_model OOF | 21,755.56 |
| v10 | postal_sector min_count=10 smoothing | _(pending)_ |
| **v11** | **+ price interaction features + hdb_age fix** | **_(submit)_** |

### Key questions after running
- Did `ft_sector_median_price` reduce OOF RMSE by >$100? → confirms interaction encoding value
- Is `yr_sector_median_price` helping or hurting? (high sparsity — check fold-by-fold RMSE variance)
- Are all Ridge coefficients positive? → confirms 3-model diversity preserved

### Next steps if v11 improves
- [ ] Try Optuna (Bayesian HPO) for smarter hyperparameter search
- [ ] Add CatBoost as 4th base model (native categorical handling)
- [ ] Explore `(flat_type, town)` OOF as a cheaper alternative to `ft_sector` if sparsity is an issue